# Approximation Algorithms for Steiner Trees in Weighted Graphs

Nikola Labus — Naučno izračunavanje 2025/26

In [16]:
import networkx as nx
import time
from itertools import combinations
from pathlib import Path

## STP Parser

In [17]:
def parse_stp(filepath):
    G = nx.Graph()
    terminals = []
    name = ""
    section = None

    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('33D32945'):
                continue

            if line.startswith('SECTION'):
                section = line.split()[1]
                continue
            if line == 'END' or line == 'EOF':
                section = None
                continue

            if section == 'Comment':
                if line.startswith('Name'):
                    name = line.split('"')[1]

            elif section == 'Graph':
                if line.startswith('Nodes'):
                    n = int(line.split()[1])
                    G.add_nodes_from(range(1, n + 1))
                elif line.startswith('E '):
                    parts = line.split()
                    u, v, w = int(parts[1]), int(parts[2]), int(parts[3])
                    G.add_edge(u, v, weight=w)

            elif section == 'Terminals':
                if line.startswith('T '):
                    terminals.append(int(line.split()[1]))

    return G, terminals, name

## Brute Force (Egzaktan algoritam)

Za svaki podskup neterminalnih čvorova proveravamo da li se terminali mogu povezati
kroz indukovani podgraf. Pamtimo minimalno razapinjuće stablo sa najmanjom težinom.

In [18]:
def brute_force_steiner(G, terminals):
    terminal_set = set(terminals)
    non_terminals = [v for v in G.nodes() if v not in terminal_set]
    best_weight = float('inf')
    best_tree = None
    subsets_checked = 0

    for k in range(len(non_terminals) + 1):
        for subset in combinations(non_terminals, k):
            subsets_checked += 1
            nodes = terminal_set | set(subset)
            subgraph = G.subgraph(nodes)

            if nx.is_connected(subgraph):
                mst = nx.minimum_spanning_tree(subgraph)
                weight = mst.size(weight='weight')
                if weight < best_weight:
                    best_weight = weight
                    best_tree = mst

    return best_tree, best_weight, subsets_checked

## MST Heuristika (Kou, Markowsky, Berman)

Konstruišemo kompletni graf od terminala gde su težine jednake najkraćim putevima u originalnom grafu.
Na tom grafu nalazimo MST, a zatim zamenjujemo grane originalnim putevima i uklanjamo cikluse i nepotrebne listove.

In [19]:
def mst_heuristic(G, terminals):
    complete = nx.Graph()
    paths = {}
    for i, t1 in enumerate(terminals):
        for t2 in terminals[i + 1:]:
            length, path = nx.single_source_dijkstra(G, t1, t2)
            complete.add_edge(t1, t2, weight=length)
            paths[(t1, t2)] = path


    mst = nx.minimum_spanning_tree(complete)

    steiner = nx.Graph()
    for u, v in mst.edges():
        key = (u, v) if (u, v) in paths else (v, u)
        path = paths[key]
        for j in range(len(path) - 1):
            w = G[path[j]][path[j + 1]]['weight']
            steiner.add_edge(path[j], path[j + 1], weight=w)

    steiner = nx.minimum_spanning_tree(steiner)

    changed = True
    while changed:
        changed = False
        leaves = [v for v in steiner.nodes() if steiner.degree(v) == 1 and v not in terminals]
        for leaf in leaves:
            steiner.remove_node(leaf)
            changed = True
    weight = steiner.size(weight='weight')
    return steiner, weight

## Mehlhorn algoritam

Koristimo Voronoi dijagram na grafu — svaki čvor se dodeljuje najbližem terminalu.
Zatim gradimo međuregionalni graf od grana koje spajaju različite regione i nalazimo MST na njemu.

In [20]:
def mehlhorn_steiner(G, terminals):
    terminal_set = set(terminals)

    voronoi = {}
    dist = {}
    for v in G.nodes():
        dist[v] = 987654321

    for t in terminals:
        dist[t] = 0
        voronoi[t] = t

    import heapq
    pq = [(0,t) for t in terminals]
    heapq.heapify(pq)

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, data in G[u].items():
            w = data['weight']
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                voronoi[v] = voronoi[u]
                heapq.heappush(pq, (dist[v], v))

    region_graph = nx.Graph()
    for u, v, data in G.edges(data=True):
        r1 = voronoi[u]
        r2 = voronoi[v]
        if r1 != r2:
            edge_weight = dist[u] + data['weight'] + dist[v]
            if not region_graph.has_edge(r1, r2) or region_graph[r1][r2]['weight'] > edge_weight:
                region_graph.add_edge(r1, r2, weight= edge_weight, via_u=u, via_v=v)

    mst = nx.minimum_spanning_tree(region_graph)

    steiner = nx.Graph()
    for r1,r2,data in mst.edges(data=True):
        u = data['via_u']
        v = data['via_v']
        for node, terminal in [(u,r1), (v, r2)]:
            path = nx.shortest_path(G, terminal, node, weight='weight')
            for j in range(len(path) - 1):
                w = G[path[j]][path[j + 1]]['weight']
                steiner.add_edge(path[j], path[j + 1], weight=w)
        steiner.add_edge(u, v, weight=G[u][v]['weight'])

    steiner = nx.minimum_spanning_tree(steiner)

    changed = True
    while changed:
        changed = False
        leaves = [v for v in steiner.nodes() if steiner.degree(v) == 1 and v not in terminal_set]
        for leaf in leaves:
            steiner.remove_node(leaf)
            changed = True
    weight = steiner.size(weight='weight')
    return steiner, weight

## Shortest Path Heuristic — SPH (Takahashi, Matsuyama)

Krećemo od jednog terminala i iterativno dodajemo najbliži preostali terminal
najkraćim putem do rastućeg stabla. Greedy pristup sa aproksimacionim faktorom 2.

In [21]:
def sph_steiner(G, terminals):
    terminal_set = set(terminals)

    in_tree = {terminals[0]}
    remaining = set(terminals[1:])
    steiner = nx.Graph()

    while remaining:
        best_path = None
        best_length = 987654321
        best_terminal = None

        for t in remaining:
            for node in in_tree:
                try:
                    length, path = nx.single_source_dijkstra(G, node, t)
                    if length < best_length:
                        best_length = length
                        best_path = path
                        best_terminal = t
                except nx.NetworkXNoPath:
                    continue

        for j in range(len(best_path) - 1):
            w = G[best_path[j]][best_path[j + 1]]['weight']
            steiner.add_edge(best_path[j], best_path[j + 1], weight=w)

        in_tree.update(best_path)
        remaining.remove(best_terminal)

    steiner = nx.minimum_spanning_tree(steiner)

    changed = True
    while changed:
        changed = False
        leaves = [v for v in steiner.nodes() if steiner.degree(v) == 1 and v not in terminal_set]
        for leaf in leaves:
            steiner.remove_node(leaf)
            changed = True
    weight = steiner.size(weight='weight')
    return steiner, weight

## Zelikovsky algoritam (11/6 aproksimacija)

Krećemo od MST heuristike i iterativno poboljšavamo rešenje. Za svaku trojku terminala
tražimo Steinerovu tačku koja ih povezuje jeftinije od trenutnih puteva u stablu (zvezda).

In [ ]:
def zelikovsky_steiner(G, terminals):
    terminal_set = set(terminals)

    steiner, weight = mst_heuristic(G, terminals)

    all_dist = dict(nx.all_pairs_dijkstra_path_length(G, weight='weight'))
    all_paths = dict(nx.all_pairs_dijkstra_path(G, weight='weight'))

    improved = True
    while improved:
        improved = False
        best_saving = 0
        best_triple = None
        best_steiner_node = None

        for i, t1 in enumerate(terminals):
            for j, t2 in enumerate(terminals[i+1:], i+1):
                for t3 in terminals[j+1:]:
                    for v in G.nodes():
                        if v in terminal_set:
                            continue

                        star_cost = all_dist[v][t1] + all_dist[v][t2] + all_dist[v][t3]

                        try:
                            path_12 = nx.shortest_path(steiner, t1, t2, weight='weight')
                            path_23 = nx.shortest_path(steiner, t2, t3, weight='weight')
                            path_13 = nx.shortest_path(steiner, t1, t3, weight='weight')
                        except nx.NetworkXNoPath:
                            continue

                        len_12 = sum(steiner[path_12[k]][path_12[k+1]]['weight'] for k in range(len(path_12)-1))
                        len_23 = sum(steiner[path_23[k]][path_23[k+1]]['weight'] for k in range(len(path_23)-1))
                        len_13 = sum(steiner[path_13[k]][path_13[k+1]]['weight'] for k in range(len(path_13)-1))

                        lengths = sorted([len_12, len_23, len_13], reverse=True)
                        current_cost = lengths[0] + lengths[1]

                        saving = current_cost - star_cost
                        if saving > best_saving:
                            best_saving = saving
                            best_triple = (t1, t2, t3)
                            best_steiner_node = v

        if best_saving > 0:
            t1,t2,t3 = best_triple
            v = best_steiner_node

            for t in [t1,t2,t3]:
                path = all_paths[v][t]
                for k in range(len(path)-1):
                    w = G[path[k]][path[k+1]]['weight']
                    steiner.add_edge(path[k], path[k+1], weight=w)

            steiner = nx.minimum_spanning_tree(steiner)
            changed = True
            while changed:
                changed = False
                leaves = [node for node in steiner.nodes() if steiner.degree(node) == 1 and node not in terminal_set]
                for leaf in leaves:
                    steiner.remove_node(leaf)
                    changed = True

            new_weight = steiner.size(weight='weight')
            if new_weight < weight:
                weight = new_weight
                improved = True

    return steiner, weight